# Task 1: rating prediction (MSE ↓, 0-ratings skipped)

**Shipped:** bias model α + β_u + β_i (ALS), reg_u 1, reg_i 5, 25 epochs, on nonzero rows.
**Val 0.5941 | public 0.5917 | tier 2.5** (3.0 ≤ 0.58, 3.5 ≤ 0.56).

**Candidate:** `StackedRating` = bias + GB regressor on [base, item-kNN residual, support,
user/item stats] → held-out half 0.5614 (−5.5%); estimated public ~0.559 (3.5 cutoff 0.56).

**Data facts:** no test timestamps; 0% of test pairs in train; 1.48% zero rows; 1.3% of test
rows from single-rating users. Error is concentrated in 1–2 star ratings (7% of rows, 36% of error).

**Open question:** leaderboard cluster at ~0.52 (tier 4.0). Ask a TA.

In [19]:
%load_ext autoreload
%autoreload 2

from assignment_1.modules.util import calc_mse, drop_zero_ratings, load_csv, random_split
from assignment_1.modules.models import BiasModel, ItemKNN, RatingStats, StackedRating, SVDModel, adjusted_cosine, jaccard
from tqdm.auto import tqdm
from sklearn.ensemble import HistGradientBoostingRegressor
import pandas as pd
import numpy as np
import itertools


DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_1_test.csv", DATA_DIR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data checks

In [8]:
def key(d): return d["user"] + "|" + d["item"]


z = train["rating"] == 0
g = train[~z].groupby("user")["rating"].nunique()
print(f"{len(train):,} rows | {train.user.nunique():,} users | {train.item.nunique():,} items")
print(
    f"test unseen: users {(~test.user.isin(train.user)).mean():.1%}, items {(~test.item.isin(train.item)).mean():.1%}")
print(
    f"test columns: {test.columns.tolist()}  (rating column is empty: the prediction slot)")
print(f"test pairs in train: {key(test).isin(set(key(train))).mean():.2%}")
print(
    f"zero rows: {z.mean():.2%} | users with any zero: {train.loc[z, 'user'].nunique() / train.user.nunique():.1%}")
print(
    f"test rows from single-rating users: {(test.user.map(g) == 1).mean():.1%}")
train.rating.value_counts().sort_index()

1,861,643 rows | 30,909 users | 51,007 items
test unseen: users 0.0%, items 0.0%
test columns: ['user', 'item', 'rating']  (rating column is empty: the prediction slot)
test pairs in train: 0.00%
zero rows: 1.48% | users with any zero: 23.7%
test rows from single-rating users: 1.3%


rating
0     27626
1     36443
2     97748
3    314624
4    641182
5    744020
Name: count, dtype: int64

## Validation split + yardstick
One split for every experiment; `BiasModel` must reproduce the shipped 0.5941.

In [9]:
tr, val = random_split(train, val_frac=0.25, seed=0)
tr, val = drop_zero_ratings(tr), drop_zero_ratings(val)
y_val = val["rating"].to_numpy()

base_model = BiasModel(reg_u=1.0, reg_i=5.0, n_epochs=25).fit(tr)
base_pred = base_model.predict(val)
BASE = calc_mse(y_val, base_pred)
print(f"shipped bias model: {BASE:.4f}  (expect 0.5941)")

shipped bias model: 0.5941  (expect 0.5941)


## Helpers
`run_grid(make, grid)`: `make(**params)` returns an unfitted `RatingModel`; each is fit on `tr`
and scored on `val`.

In [10]:
def run_grid(make, grid: dict) -> pd.DataFrame:
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        mse = calc_mse(y_val, make(**params).fit(tr).predict(val))
        rows.append({**params, "val_mse": mse,
                    "vs_base_%": (1 - mse / BASE) * 100})
    return pd.DataFrame(rows).sort_values("val_mse").reset_index(drop=True)


def svd_sep(k, reg_b, reg_f, lr=0.005, epochs=20) -> SVDModel:
    """SVD with separate bias / factor regularization (SGD scale)."""
    return SVDModel(n_factors=k, reg_bu=reg_b, reg_bi=reg_b, reg_pu=reg_f, reg_qi=reg_f,
                    lr_all=lr, n_epochs=epochs)

## E1. Bias model λ (shipped)
**Result:** best reg_u 1, reg_i 5 (inside the grid) → 0.5941. More epochs: no change.

In [11]:
# run_grid(BiasModel, {"reg_u": [0.3, 1, 3], "reg_i": [2, 5, 10], "n_epochs": [25]})

## E2. SVD with one λ (`reg_all`)
**Result:** best k = 1, reg_all 0.05, 50 epochs → 0.5989. More factors are worse at every λ;
50 epochs beats 20 (SGD not converged at 20).

In [12]:
# run_grid(lambda **p: SVDModel(**p),
#          {"n_factors": [1, 10, 50], "reg_all": [0.02, 0.05, 0.1, 0.3], "n_epochs": [20, 50]})

## E3. SVD with separate bias / factor λ + convergence
**Result:** best 0.5980 (k 5, reg_b 0.005, reg_f 0.3, lr 0.003, 60 epochs). Convergence check:
lr 0.002 / 120 epochs → 0.5972 (best); 200 epochs slightly worse (overfits). SGD is converged;
the factor model's ceiling is ~0.597, worse than bias. Factors don't help on this data
(same as PML ch. 5 exercise: LFM 1.0093 vs bias-only 1.0048).

In [13]:
# run_grid(svd_sep, {"k": [5], "reg_b": [0.005], "reg_f": [0.3],
#                    "lr": [0.002, 0.003], "epochs": [60, 120, 200]})

## E4. Where is the bias model's error?
**Result:** 1–2 star ratings are 7% of rows but 36% of error (MSE 5.06 / 2.09); error grows
with the user's rating std; heavy users (100+) carry 45%; no cold-start problem.
→ missing signal: user-specific dislikes (taste toward particular books).

In [14]:
err = (base_pred - y_val) ** 2
n_u = val["user"].map(tr["user"].value_counts()).fillna(0).to_numpy()
u_std = val["user"].map(tr.groupby(
    "user")["rating"].std()).fillna(0).to_numpy()


def err_by(x, bins, name):
    t = (pd.DataFrame({"b": pd.cut(x, bins, include_lowest=True), "e": err})
           .groupby("b", observed=True)["e"].agg(["mean", "size", "sum"]))
    t["share_of_error"] = t["sum"] / err.sum()
    print(f"\n{name}\n{t[['mean', 'size', 'share_of_error']].round(3)}")


err_by(n_u, [0, 1, 3, 10, 30, 100, 1e6], "user history size")
err_by(u_std, [0, 0.01, 0.5, 0.8, 1.0, 1.3, 5], "user rating std")
err_by(y_val, [0.5, 1.5, 2.5, 3.5, 4.5, 5.5], "true rating")


user history size
                     mean    size  share_of_error
b                                                
(-0.001, 1.0]       0.634      16           0.000
(1.0, 3.0]          0.617     176           0.000
(3.0, 10.0]         0.779   26201           0.075
(10.0, 30.0]        0.708   74471           0.194
(30.0, 100.0]       0.629  122315           0.282
(100.0, 1000000.0]  0.519  235290           0.449

user rating std
                 mean    size  share_of_error
b                                            
(-0.001, 0.01]  0.135    7324           0.004
(0.01, 0.5]     0.204   56784           0.043
(0.5, 0.8]      0.433  198850           0.316
(0.8, 1.0]      0.717  109946           0.289
(1.0, 1.3]      1.039   71769           0.274
(1.3, 5.0]      1.476   13796           0.075

true rating
               mean    size  share_of_error
b                                          
(0.499, 1.5]  5.060    9082           0.169
(1.5, 2.5]    2.088   24542           0.188
(2.5, 3

## E5. Item-kNN on bias residuals (`ItemKNN`)
r̂ = b̂ + a · Σ cos(i,j)(r_uj − b̂_uj) / (λ + Σ cos(i,j)), cosine from co-reading.
**Result:** λ 0.5 → 0.5706 (−4.0%); tuned λ 2, a 2.0 → 0.5674 (−4.5%). a > 1 helps → the
correction is diluted over long histories (top-k is the fix). λ = 0 gives NaN (0/0 rows).

In [15]:
knn = ItemKNN().fit(tr, base_model)
num, den = knn.parts(val)
for lam, a in [(0.5, 1.0), (1, 1.5), (2, 2.0)]:
    print(f"λ={lam:<4} a={a:<4} MSE {calc_mse(y_val, np.clip(base_pred + a * num / (den + lam), 1, 5)):.4f}")

λ=0.5  a=1.0  MSE 0.5706
λ=1    a=1.5  MSE 0.5680
λ=2    a=2.0  MSE 0.5674


## E6. Stacked: GB regressor on the bias residual (held-out half)
Features: base, knn, log_support (`ItemKNN`), log_n_u, u_std, log_n_i (`RatingStats`).
**Result:** bias 0.5941 | kNN hand-tuned 0.5674 | stacked **0.5614** (−5.5%).

In [16]:
stats = RatingStats().fit(tr, base_model)
F = pd.concat([pd.DataFrame({"base": base_pred}),
              knn.features(val), stats.features(val)], axis=1)

h = np.random.default_rng(0).random(
    len(y_val)) < 0.5      # fit on h, score on ~h
reg = HistGradientBoostingRegressor(
    random_state=0).fit(F[h], y_val[h] - base_pred[h])
stacked = np.clip(base_pred + reg.predict(F), 1, 5)
hand = np.clip(base_pred + 2.0 * num / (den + 2.0), 1, 5)
ho = ~h
pd.Series({
    "bias": calc_mse(y_val[ho], base_pred[ho]),
    "kNN hand-tuned": calc_mse(y_val[ho], hand[ho]),
    "stacked": calc_mse(y_val[ho], stacked[ho]),
}).round(4)

bias              0.5941
kNN hand-tuned    0.5674
stacked           0.5614
dtype: float64

## E7. `StackedRating` class: outer-holdout check
The class trains its regressor on its own internal split, so score it on an outer holdout it
never sees. Expect ~5% below the bias model, as in E6.

In [17]:
nonzero = drop_zero_ratings(train)
outer_tr, outer_val = random_split(nonzero, val_frac=0.1, seed=1)
y_out = outer_val["rating"].to_numpy()

stacked_model = StackedRating(
    base=BiasModel(reg_u=1.0, reg_i=5.0),
    sources=[ItemKNN(), RatingStats()],
    regressor=HistGradientBoostingRegressor(random_state=0),
).fit(outer_tr)

pd.Series({
    "bias": calc_mse(y_out, BiasModel(reg_u=1.0, reg_i=5.0).fit(outer_tr).predict(outer_val)),
    "stacked (class)": calc_mse(y_out, stacked_model.predict(outer_val)),
}).round(4)

bias               0.5876
stacked (class)    0.5514
dtype: float64

## E8. Base model swap: SVD instead of bias inside the stack
Same stack with the best SVD (E3) as the base. Expect no gain (factors ≈ no signal), but it
checks that the stack works with any `RatingModel`.

In [18]:
# svd_stack = StackedRating(
#     base=svd_sep(5, 0.005, 0.3, lr=0.002, epochs=120),
#     sources=[ItemKNN(), RatingStats()],
#     regressor=HistGradientBoostingRegressor(random_state=0),
# ).fit(outer_tr)
# calc_mse(y_out, svd_stack.predict(outer_val))

## E9. Similarity measures for the kNN source (outer holdout)
Co-reading (binary cosine, shipped) vs agreement (adjusted cosine on bias residuals, shrunk
by co-support) vs both as separate sources. Baseline: E7 stacked 0.5514; 4.0 needs about
0.541 on this holdout (−1.8%).

In [20]:
variants = {
    "cosine (shipped)": [ItemKNN(), RatingStats()],
    "jaccard": [ItemKNN(similarity=jaccard), RatingStats()],
    "adjusted cosine": [ItemKNN(similarity=adjusted_cosine), RatingStats()],
    "cosine + adjusted": [ItemKNN(prefix="knn_cos"),
                          ItemKNN(similarity=adjusted_cosine,
                                  prefix="knn_adj"),
                          RatingStats()],
}
e9 = {}
for name, sources in tqdm(variants.items()):
    m = StackedRating(
        BiasModel(reg_u=1.0, reg_i=5.0),
        sources,
        HistGradientBoostingRegressor(random_state=0)
    ).fit(outer_tr)
    e9[name] = calc_mse(y_out, m.predict(outer_val))
    print(f"{name:<20} {e9[name]:.4f}")

  0%|          | 0/4 [00:00<?, ?it/s]

cosine (shipped)     0.5514
jaccard              0.5492
adjusted cosine      0.5640
cosine + adjusted    0.5477


In [21]:
for params in [dict(), dict(max_iter=300, learning_rate=0.05),
               dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=63)]:
    m = StackedRating(BiasModel(reg_u=1.0, reg_i=5.0),
                      [ItemKNN(similarity=jaccard), RatingStats()],
                      HistGradientBoostingRegressor(random_state=0, **params)).fit(outer_tr)
    print(params, f"{calc_mse(y_out, m.predict(outer_val)):.4f}")

{} 0.5492
{'max_iter': 300, 'learning_rate': 0.05} 0.5491
{'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 63} 0.5493


In [22]:
tests = {
    "jaccard + adjusted": [ItemKNN(similarity=jaccard, prefix="knn_jac"),
                           ItemKNN(similarity=adjusted_cosine, prefix="knn_adj"), RatingStats()],
    "jaccard + adjusted, tuned GB": None,
}
srcs = tests["jaccard + adjusted"]
for name, gb in [("jaccard + adjusted", {}),
                 ("jaccard + adjusted, tuned GB", dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=63))]:
    m = StackedRating(BiasModel(reg_u=1.0, reg_i=5.0), srcs,
                      HistGradientBoostingRegressor(random_state=0, **gb)).fit(outer_tr)
    print(f"{name:<30} {calc_mse(y_out, m.predict(outer_val)):.4f}")

jaccard + adjusted             0.5465
jaccard + adjusted, tuned GB   0.5466


## Next
- If E7 confirms ~5%: ship `StackedRating` in `run_task1`, `make check`, upload (aim ≤ 0.56 public).
- Margin for 3.5: residual (adjusted-cosine) similarity and top-k kNN as extra `FeatureSource`s.
- Port the same pipeline to Task 3 (with `TimeBiasModel` as the base).